---
title: "最短路径——Floyd-Warshall 算法"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

In [2]:
#| code-fold: true
import math
from typing import List

## 📝 题目 1334：阈值距离内邻居最少的城市

::: {.callout-caution}
### 📖 题目描述
有 `n` 个城市，按从 `0` 到 `n-1` 编号。给你一个边数组 `edges`，其中 `edges[i] =` [$from_i$, $to_i$, $weight_i$] 代表 $from_i$ 和 $to_i$ 两个城市之间有一条**双向**边，距离为 $weight_i$。
同时给你一个整数 `distanceThreshold`（距离阈值）。

一个城市的“可达邻居数”，是指那些**最短路径距离不超过 `distanceThreshold`** 的城市数量。

请你返回**可达邻居数最少**的城市。
**注意：**如果有多个城市的可达邻居数一样少，请返回**编号最大**的那个城市。

**输入输出示例**：

* **示例 1**：
    * **输入**：`n = 4`, `edges = [[0,1,3],[1,2,1],[1,3,4],[2,3,1]]`, `distanceThreshold = 4`
    * **输出**：`3`
    * **解释**：
      城市 0 的可达城市有：1, 2 (共 2 个)
      城市 1 的可达城市有：0, 2, 3 (共 3 个)
      城市 2 的可达城市有：0, 1, 3 (共 3 个)
      城市 3 的可达城市有：1, 2 (共 2 个)
      城市 0 和 3 的可达城市一样少（都是 2 个）。题目要求返回编号最大的，所以返回 3。

* **示例 2**：
    * **输入**：`n = 5`, `edges = [[0,1,2],[0,4,8],[1,2,3],[1,4,2],[2,3,1],[3,4,1]]`, `distanceThreshold = 2`
    * **输出**：`0`
    * **解释**：
      城市 0 的可达城市只有自己（距离<=2内没有别人），共 0 个。（注：自身不算在邻居内）
      ...计算其他城市后，发现城市 0 的邻居最少，返回 0。
:::

In [5]:
class Solution1334:
    def findTheCity(self, n: int, edges: List[List[int]], distanceThreshold: int) -> int:
        dist = [[math.inf] * n for _ in range(n)]  # 建立 N x N 的二维距离矩阵
        for i in range(n):  # 自己到自己的距离永远是 0
            dist[i][i] = 0
        for edge in edges:  # 录入直达航班的数据（无向图，双向写入）
            dist[edge[0]][edge[1]] = edge[2]
            dist[edge[1]][edge[0]] = edge[2]
        for k in range(n):  # Floyd-Warshall 核心
            for i in range(n):
                for j in range(n):
                   if dist[i][k] + dist[k][j] < dist[i][j]:  # 如果经过 k 中转更近，就更新距离表
                       dist[i][j] = dist[i][k] + dist[k][j]
        min_reachable = math.inf
        best_city = -1
        for i in range(n):
            count = 0
            for j in range(n):
                if i != j and dist[i][j] <= distanceThreshold:  # 如果不是自己，且距离在阈值内，邻居数 +1
                    count += 1
            if count <= min_reachable:
                min_reachable = count
                best_city = i
        return best_city

In [6]:
#| code-fold: true
def test_1334(func):
    test_cases = [
        {"desc": "示例 1: 常规网络", "n": 4, "edges": [[0,1,3],[1,2,1],[1,3,4],[2,3,1]], "distanceThreshold": 4, "expected": 3},
        {"desc": "示例 2: 包含孤岛和长链", "n": 5, "edges": [[0,1,2],[0,4,8],[1,2,3],[1,4,2],[2,3,1],[3,4,1]], "distanceThreshold": 2, "expected": 0},
        {"desc": "全互不连通 (都为0个邻居，返回最大ID)", "n": 4, "edges": [], "distanceThreshold": 10, "expected": 3},
        {"desc": "阈值极大 (都能到达，返回最大ID)", "n": 3, "edges": [[0,1,1],[1,2,1]], "distanceThreshold": 100, "expected": 2},
        {"desc": "阈值极小 (谁也到不了谁，返回最大ID)", "n": 3, "edges": [[0,1,10],[1,2,10]], "distanceThreshold": 1, "expected": 2},
        {"desc": "星型网络 (中心点邻居多，边缘点少)", "n": 4, "edges": [[0,1,1],[0,2,1],[0,3,1]], "distanceThreshold": 1, "expected": 3}, # 1, 2, 3 都只有 1 个邻居 (0号)，取最大的 3
        {"desc": "直线形网络 (边界点邻居最少)", "n": 5, "edges": [[0,1,1],[1,2,1],[2,3,1],[3,4,1]], "distanceThreshold": 2, "expected": 4},
        {"desc": "绕远路比直达更近 (SPFA 必测)", "n": 3, "edges": [[0,1,10],[0,2,1],[1,2,1]], "distanceThreshold": 3, "expected": 2}, # 0->2->1 距离2. 所有点互相距离<=3，都连通，选最大 2
        {"desc": "两个完全相同的独立网络 (返回后面那个)", "n": 4, "edges": [[0,1,1],[2,3,1]], "distanceThreshold": 2, "expected": 3},
        {"desc": "刚好卡在阈值线上", "n": 3, "edges": [[0,1,5],[1,2,5]], "distanceThreshold": 5, "expected": 2}, # 0邻居[1], 1邻居[0,2], 2邻居[1]. 0和2最少，返回2
        {"desc": "极限: 单个城市", "n": 1, "edges": [], "distanceThreshold": 1, "expected": 0}
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'实际':<8} | {'描述'}")
    print("-" * 75)

    for i, tc in enumerate(test_cases):
        actual = func(tc["n"], tc["edges"], tc["distanceThreshold"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual:<8} |{tc['desc']}")

    print("-" * 75)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_1334(Solution1334().findTheCity)

ID   | 结果     | 预期   | 实际       | 描述
---------------------------------------------------------------------------
1    | ✅ PASS | 3    | 3        |示例 1: 常规网络
2    | ✅ PASS | 0    | 0        |示例 2: 包含孤岛和长链
3    | ✅ PASS | 3    | 3        |全互不连通 (都为0个邻居，返回最大ID)
4    | ✅ PASS | 2    | 2        |阈值极大 (都能到达，返回最大ID)
5    | ✅ PASS | 2    | 2        |阈值极小 (谁也到不了谁，返回最大ID)
6    | ✅ PASS | 3    | 3        |星型网络 (中心点邻居多，边缘点少)
7    | ✅ PASS | 4    | 4        |直线形网络 (边界点邻居最少)
8    | ✅ PASS | 2    | 2        |绕远路比直达更近 (SPFA 必测)
9    | ✅ PASS | 3    | 3        |两个完全相同的独立网络 (返回后面那个)
10   | ✅ PASS | 2    | 2        |刚好卡在阈值线上
11   | ✅ PASS | 0    | 0        |极限: 单个城市
---------------------------------------------------------------------------
测试总结: 通过 11/11


::: {.callout-important}
### 💡 思路讲解

当我们面临“求出全地图所有点到所有点的最短路径”这种多源最短路需求，且城市数量 $N$ 较小（通常 $N \le 400$）时，Floyd-Warshall 是当之无愧的王者。

**⚔️ 核心三步走：**

1. **基建（构建初始认知）**：

   - 建立一张 $N \times N$ 的二维矩阵表 `dist`，全部填上无穷大。
   - 规定自己到自己的距离为 0 (`dist[i][i] = 0`)。
   - 把已知的直达航班（`edges`）直接填进表里。此时的表，代表着“如果不允许任何中转，各城市间的距离”。

2. **核心引擎（开放中转站）**：

   - 这是算法的灵魂！我们要依次开放每一个城市 $k$ 作为“全人类的中转站”。
   - 对于全图任意两个城市 $i$ 和 $j$，我们要进行一次极其直击灵魂的审问：
     **“如果我从 $i$ 到 $j$，先去 $k$ 绕一圈，会不会比我现在记录的路线更近？”**
   - 状态转移方程：`dist[i][j] = min(dist[i][j], dist[i][k] + dist[k][j])`

3. **上帝阅卷（暴力查表）**：

   - 4 行代码跑完后，`dist` 矩阵就已经进化成了“绝对的最短距离全知字典”。
   - 我们只需遍历字典的每一行，数一数有多少个距离 $\le \text{distanceThreshold}$。
   - 遇到一样小的，直接让行号（城市编号）更大的上位。

4. **为什么 `k` 必须放在最外层循环？**

因为这是动态规划的“阶段 (Stage)”！我们是在计算“当允许经过前 $k$ 个节点作为中转时，任意两点的最短距离”。如果把 `k` 放里面，相当于你只针对一对具体的 $i$ 和 $j$ 去找中转站，而没有利用到其他点之间已经优化好的路线。**必须先开放政策（更新 $k$），再让全民去适应（遍历 $i$ 和 $j$）！**
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(V^3)$。三个嵌套的 `for` 循环（$k, i, j$），没有任何多余废话。非常适合节点数 $V \le 400$ 的稠密图。
* **空间复杂度**：$O(V^2)$。只需要开辟一张 $V \times V$ 的二维矩阵表来存储任意两点间的距离。
:::

## 📝 题目 2976：转换字符串的最小成本 I

::: {.callout-caution}
### 📖 题目描述
给你两个长度相同的字符串 `source` 和 `target`，它们仅由小写英文字母组成。
另给你两个字符数组 `original` 和 `changed`，以及一个整数数组 `cost`，其中 `cost[i]` 代表将字符 `original[i]` 更改为字符 `changed[i]` 的成本。

你需要将字符串 `source` 转化为字符串 `target`。每一次转化，你可以将 `source` 中的一个字符变成另一个字符，只要这个转换规则在数组里，并且支付相应的 `cost`。
*(注意：你可以对同一个字符进行多次中转转换，比如 'a' -> 'b' -> 'c')*

请你返回将 `source` 转化为 `target` 的 **最小总成本**。如果无法转化，返回 `-1`。

**输入输出示例**：

* **示例 1**：
    * **输入**：`source = "abcd"`, `target = "acbe"`, `original = ["a","b","c","c","e","d"]`, `changed = ["b","c","b","e","b","e"]`, `cost = [2,5,5,1,2,20]`
    * **输出**：`28`
    * **解释**：
      - 'a' 变成 'a'，成本 0。
      - 'b' 变成 'c'，成本 5。
      - 'c' 变成 'b'，成本 5。
      - 'd' 变成 'e'，我们要找最便宜的路线。可以直接 'd'->'e'(成本20)。但如果我们绕路走 'd'->'e' 的其他可能？这题直接给的 'd'->'e' 是 20。等等，有没有更便宜的？如果没有，就花 20。总成本 0 + 5 + 5 + 20 = 30？不对，题目样例的输出是 28。
      *(真实路线揭秘：'d' 不能直接变，'d' 变成 'e' 的成本是 20。但在其他数据里，如果有 'd'->'b'(成本2) + 'b'->'e'(成本3) 呢？我们要找的是全图的最短路！)*

* **示例 2**：
    * **输入**：`source = "aaaa"`, `target = "bbbb"`, `original = ["a","c"]`, `changed = ["c","b"]`, `cost = [1,2]`
    * **输出**：`12`
    * **解释**：要把 'a' 变成 'b'，没有直达路线。只能先 'a' -> 'c'(成本1)，再 'c' -> 'b'(成本2)。所以单次转换成本是 3。一共 4 个 'a'，总成本 3 * 4 = 12。
:::

In [9]:
class Solution2976:
    def minimumCost(self, source: str, target: str, original: List[str], changed: List[str], cost: List[int],) -> int:
        dist = [[math.inf] * 26 for _ in range(26)]  # 26字母转换矩阵
        for i in range(26):  # 自己变自己，成本为 0
            dist[i][i] = 0
        for i in range(len(original)):  # 录入规则 (防止劣质报价覆盖优质报价)
            start = ord(original[i]) - ord('a')
            end = ord(changed[i]) - ord('a')
            dist[start][end] = min(cost[i], dist[start][end])
        for k in range(26):  # Floyd 核心
            for i in range(26):
                for j in range(26):
                    if dist[i][k] + dist[k][j] < dist[i][j]:
                        dist[i][j] = dist[i][k] + dist[k][j]
        total_cost = 0  # 算总账
        for i in range(len(source)):
            if dist[ord(source[i]) - ord('a')][ord(target[i]) - ord('a')] == math.inf:  # 如果遇到不可能的任务，直接返回 -1
                return -1
            total_cost += dist[ord(source[i]) - ord('a')][ord(target[i]) - ord('a')]
        return total_cost

In [10]:
#| code-fold: true
def test_2976_ultimate(func):
    test_cases = [
        {"desc": "示例 1: 复杂中转", "source": "abcd", "target": "acbe", "original": ["a","b","c","c","e","d"], "changed": ["b","c","b","e","b","e"], "cost": [2,5,5,1,2,20], "expected": 28},
        {"desc": "示例 2: 多个相同字符中转", "source": "aaaa", "target": "bbbb", "original": ["a","c"], "changed": ["c","b"], "cost": [1,2], "expected": 12},
        {"desc": "示例 3: 无法转化", "source": "abcd", "target": "abce", "original": ["a"], "changed": ["e"], "cost": [10000], "expected": -1},
        {"desc": "平行边博弈 (取最小)", "source": "a", "target": "b", "original": ["a", "a"], "changed": ["b", "b"], "cost": [10, 2], "expected": 2},
        {"desc": "有向图防逆行", "source": "b", "target": "a", "original": ["a"], "changed": ["b"], "cost": [5], "expected": -1},
        {"desc": "完全不需要转换", "source": "hello", "target": "hello", "original": ["h"], "changed": ["e"], "cost": [1], "expected": 0},
        {"desc": "长链传递寻路", "source": "a", "target": "e", "original": ["a","b","c","d"], "changed": ["b","c","d","e"], "cost": [1,1,1,1], "expected": 4},
        {"desc": "两个不连通的孤岛图", "source": "a", "target": "d", "original": ["a","c"], "changed": ["b","d"], "cost": [1,2], "expected": -1},
        {"desc": "环形网络博弈 (避开高价直达)", "source": "a", "target": "c", "original": ["a","b","c","a"], "changed": ["b","c","a","c"], "cost": [1,1,1,10], "expected": 2}, # a->b->c(2) 比直达 a->c(10) 便宜
        {"desc": "陷阱: 自己变自己还要收天价", "source": "a", "target": "a", "original": ["a"], "changed": ["a"], "cost": [10000], "expected": 0}, # 考验 dist[i][i]=0 是否被错误覆盖
        {"desc": "超长字符串但无法中转", "source": "zzzzzz", "target": "yyyyyy", "original": ["z","y"], "changed": ["x","x"], "cost": [1,1], "expected": -1}
    ]

    passed = 0
    print(f"{'ID':<3} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 75)

    for i, tc in enumerate(test_cases):
        actual = func(tc["source"], tc["target"], tc["original"], tc["changed"], tc["cost"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<3} | {status:<6} | {tc['expected']:<4} | {actual:<4} | {tc['desc']}")

    print("-" * 75)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_2976_ultimate(Solution2976().minimumCost)

ID  | 结果     | 预期   | 实际   | 描述
---------------------------------------------------------------------------
1   | ✅ PASS | 28   | 28   | 示例 1: 复杂中转
2   | ✅ PASS | 12   | 12   | 示例 2: 多个相同字符中转
3   | ✅ PASS | -1   | -1   | 示例 3: 无法转化
4   | ✅ PASS | 2    | 2    | 平行边博弈 (取最小)
5   | ✅ PASS | -1   | -1   | 有向图防逆行
6   | ✅ PASS | 0    | 0    | 完全不需要转换
7   | ✅ PASS | 4    | 4    | 长链传递寻路
8   | ✅ PASS | -1   | -1   | 两个不连通的孤岛图
9   | ✅ PASS | 2    | 2    | 环形网络博弈 (避开高价直达)
10  | ✅ PASS | 0    | 0    | 陷阱: 自己变自己还要收天价
11  | ✅ PASS | -1   | -1   | 超长字符串但无法中转
---------------------------------------------------------------------------
测试总结: 通过 11/11


::: {.callout-important}
### 💡 思路讲解

这道题表面上是字符串操作，本质上是一张**“26个字母的交通网络图”**！

- **节点**：26 个小写字母（可以映射成 `0` 到 `25` 的索引）。
- **有向边**：`original[i]` 指向 `changed[i]`，权重是 `cost[i]`。
- **目标**：我们需要频繁查询从字母 $X$ 到字母 $Y$ 的最低成本。

因为我们需要知道**任意两个字母之间的最低转换成本**，且总节点数 $V = 26$ 极小，这简直就是给 **Floyd-Warshall** 量身定制的舞台！

**算法流程**：

1. **基建（构建 26x26 字典）**：

   - 建立 `dist = [[math.inf] * 26 for _ in range(26)]`。
   - `dist[i][i] = 0`。
   - 遍历 `original`, `changed`, `cost`，把给定的转换规则填入矩阵。**（注意：可能有平行边，比如两个不同的魔法都能把 'a' 变成 'b'，我们必须取 `min`）**。

2. **Floyd 4行引擎**：

   - 循环 `k, i, j` 从 0 到 25。
   - `dist[i][j] = min(dist[i][j], dist[i][k] + dist[k][j])`。

3. **查字典算总账**：

   - 遍历 `source` 和 `target` 的每一个对应字符 `src_char` 和 `tgt_char`。
   - 从 `dist` 矩阵中直接 $O(1)$ 查出 `dist[src_char][tgt_char]` 的成本，累加到 `total_cost`。
   - 如果查到是无穷大，说明这个字母根本变不过去，直接 `return -1`！
:::

::: {.callout-note}
### 💡 复杂度分析
假设字符串 `source` 的长度为 $L$。字母表大小 $V = 26$。
* **时间复杂度**：$O(V^3 + L)$。Floyd 算法需要 $O(26^3)$ 的常数级时间，瞬间跑完。之后只需遍历一次长度为 $L$ 的字符串，每次查表 $O(1)$。
* **空间复杂度**：$O(V^2) = O(26^2)$。只需要一个 $26 \times 26$ 的常数级矩阵。极其省空间。
:::